# Faruq-v3 — validation-only error diagnosis

Mendiagnosis proposal, lokalisasi, dan salah klasifikasi pada checkpoint D0 yang sudah selesai. Notebook ini tidak melakukan training dan tidak mengakses test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)


In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_diagnostic.json'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('CHECKPOINT:', CHECKPOINT)
print('OUTPUT    :', OUTPUT)


In [ ]:
command = [
    sys.executable, '-u', '-m',
    'coffee_detector.analysis.faruq_v3_diagnostics',
    '--checkpoint', str(CHECKPOINT),
    '--data-root', str(DATA_ROOT),
    '--output', str(OUTPUT),
    '--split', 'val',
    '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import json, pandas as pd
from IPython.display import display

result = json.loads(OUTPUT.read_text(encoding='utf-8'))
assert result['training_executed'] is False
assert result['test_images_accessed'] is False
global_table = pd.DataFrame([result['global']])
class_table = pd.DataFrame(result['per_class'])
confusion_table = pd.DataFrame(result['top_directional_confusions'])
percent = {
    key: '{:.2%}' for key in [
        'proposal_accessibility', 'matched_recall',
        'localization_conditioned_class_accuracy',
        'oracle_class_accuracy_headroom',
    ] if key in global_table.columns
}
display(global_table.style.format(percent))
display(class_table.style.format({
    'proposal_accessibility': '{:.2%}',
    'matched_recall': '{:.2%}',
    'localization_conditioned_class_accuracy': '{:.2%}',
}))
display(confusion_table)
print('SUMMARY:', OUTPUT)
print('Kirim tabel global, lima kelas terbawah, dan confusion pairs. Jangan training model baru.')
